# Step 1: Launch SageMaker Processing Job

Uses `ScriptProcessor` with the sklearn container to run `preprocess.py`.
Input: raw data in S3. Output: train/test CSVs in S3.

## Configuration

In [ ]:
import boto3
import sagemaker

REGION = boto3.session.Session().region_name
sess = sagemaker.session.Session()
BUCKET = sess.default_bucket()
S3_PREFIX = "autogluon-tabular"

S3_INPUT = f"s3://{BUCKET}/{S3_PREFIX}/raw/"
S3_OUTPUT = f"s3://{BUCKET}/{S3_PREFIX}/processed"
INSTANCE_TYPE = "ml.m5.xlarge"

## Discover IAM role

In [ ]:
def get_role(role_arn=None):
    if role_arn:
        return role_arn
    iam = boto3.client("iam")
    paginator = iam.get_paginator("list_roles")
    for page in paginator.paginate():
        for role in page["Roles"]:
            if "SageMaker" in role["RoleName"] or "sagemaker" in role["RoleName"]:
                print(f"Discovered role: {role['Arn']}")
                return role["Arn"]
    raise ValueError("No SageMaker IAM role found. Pass role_arn explicitly.")

role_arn = get_role()

## Set up the ScriptProcessor

In [ ]:
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.processing import ProcessingInput, ProcessingOutput, ScriptProcessor
from sagemaker.core.shapes.shapes import ProcessingS3Input, ProcessingS3Output

session = Session()

processing_image = image_uris.retrieve("sklearn", region=REGION, version="1.2-1")
print(f"Processing image: {processing_image}")

processor = ScriptProcessor(
    image_uri=processing_image,
    role=role_arn,
    command=["python3"],
    instance_type=INSTANCE_TYPE,
    instance_count=1,
    sagemaker_session=session,
)

## Run the processing job

In [ ]:
s3_output = S3_OUTPUT.rstrip("/")

processor.run(
    code="preprocess.py",
    inputs=[
        ProcessingInput(
            input_name="input",
            s3_input=ProcessingS3Input(
                s3_uri=S3_INPUT,
                local_path="/opt/ml/processing/input",
                s3_data_type="S3Prefix",
            ),
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            s3_output=ProcessingS3Output(
                s3_uri=f"{s3_output}/train/",
                local_path="/opt/ml/processing/train",
                s3_upload_mode="EndOfJob",
            ),
        ),
        ProcessingOutput(
            output_name="test",
            s3_output=ProcessingS3Output(
                s3_uri=f"{s3_output}/test/",
                local_path="/opt/ml/processing/test",
                s3_upload_mode="EndOfJob",
            ),
        ),
    ],
    wait=True,
    logs=True,
)

print(f"\nProcessing complete.")
print(f"Train data: {s3_output}/train/")
print(f"Test data:  {s3_output}/test/")
print("Next: run 1-training/launch_training.ipynb with these S3 paths.")